In [ ]:
!pip install git+https://github.com/openai/whisper.git
!sudo apt update -y && sudo apt install ffmpeg -y
!pip install datasets
!pip install torch
!pip install tqdm
!pip install pandas

In [ ]:
import whisper
import torch  # pytorch install steps: pytorch.org

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = whisper.load_model("large").to(device)

In [ ]:
from pathlib import Path
import os
# Shared folder with Step5_Scrape_Youtube's MP3 conversion output.
PROJECT_DIR = Path(os.environ.get('SJK_PORTFOLIO_DIR', '/content/sjk_portfolio'))
AUDIO_DIR = PROJECT_DIR / 'audio' / 'mp3_converted'
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
# To use Google Drive instead, mount it manually and set SJK_PORTFOLIO_DIR.


In [ ]:
from datasets import load_dataset

videos_meta = load_dataset(
    "jamescalam/channel-metadata",
    split="train"
)
videos_meta

In [ ]:
from datasets import load_dataset

videos_meta = load_dataset("jamescalam/channel-metadata", split="train")

# Convert the data to a list of lines
lines = str(videos_meta).split("\n")

# Print the first 100 lines
for i, line in enumerate(lines[:100]):
    print(f"{i+1}. {line}")


In [ ]:
import pandas as pd
# df = pd.read_csv(str(PROJECT_DIR / 'audio' / 'metadata.csv'))
df = pd.read_csv(str(PROJECT_DIR / 'audio' / 'metadata.csv'))
# video_meta = df.to_dict('index')
# video_meta

In [ ]:
# videos_meta_typingagent --> CSV
# Video ID has to match .mp3 file

#Example
#videos_meta_typingagent = {'videoID': 'wwJbXDkvR_A'}

# videos_dict = {'wwJbXDkvR_A': {'title': "" , 'published': "", 'url', ""}}
videos_dict = {}


for row in df.iterrows():

# for row in videos_meta_typingagent:
    # create entry in dict
  videos_dict[row[1]['Video ID']] = {
        'title': row[1]['Title'],
        'channelId': row[1]['Channel ID'],
        'description': row[1]['Description'],
        'length(s)': row[1]['Length (sec)'],
        'date_uploaded': row[1]['Date Uploaded']
        }


    #     row[]] = {
    #     'title': row['Title'],
    #     'published': row['Time Published'],
    #     'url': f"https://youtu.be/{row['Video ID']}"
    # }

In [ ]:
import os
from datasets import load_dataset
from pathlib import Path

# Load the dataset
videos_meta = load_dataset("jamescalam/channel-metadata", split="train")

# Create the directory if it doesn't already exist
if not AUDIO_DIR.exists():
    AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Save the dataset to disk
with open(AUDIO_DIR / "dataset.txt", "w") as f:
    f.write(str(videos_meta))

# Get list of MP3 audio files
paths = [str(x) for x in Path(str(AUDIO_DIR)).glob('*.mp3')]
print(len(paths))
print(paths[:5])


In [ ]:
from pathlib import Path

# get list of MP3 audio files
paths = [str(x) for x in Path(str(AUDIO_DIR)).glob('*.mp3')]
# paths = [str(x) for x in Path(str(AUDIO_DIR)).glob('*.mp3')]
print(len(paths))
# print(paths[:5])

# paths = [paths[1]]
# paths

In [ ]:
# we get the IDs like so
for item in paths:
  print(item.split('/')[-1][:-4])

In [ ]:
from tqdm.auto import tqdm
data = []
for i, path in enumerate(tqdm(paths)):
    _id = path.split('/')[-1][:-4]
    # transcribe to get speech-to-text data
    result = model.transcribe(path)
    # add results to data list
    data.extend(result['segments'])

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import json

# set window (length of text chunk) and stride
window = 10
stride = 3  # smaller stride creates overlap

data = []

results = []
with open(PROJECT_DIR / "transcription.jsonl", "w", encoding="utf-8") as fp:
    for i, path in enumerate(tqdm(paths)):
        _id = path.split('/')[-1][:-4]
        print(_id)
        # transcribe to get speech-to-text data
        result = model.transcribe(path)
        segments = result['segments']
        if not segments:
            continue

        # video_meta = new_videos_dict[_id]
        video_meta = videos_dict.get(_id, {})
        print(video_meta)

        for j in range(0, len(segments), stride):
            j_end = min(j + window, len(segments))
            text = ' '.join(x["text"].strip() for x in segments[j:j_end])
            start = segments[j]['start']
            end = segments[j_end - 1]['end']
            row_id = f"{_id}-t{segments[j]['start']}"
            # This is where the code breaks, trying to merge the cells.
            meta = {
                **video_meta,
                **{
                    "id": row_id,
                    "text": text.strip(),
                    "start": start,
                    "end": end
                }
            }
            data.append(meta)
            json.dump(meta, fp)
            fp.write('\n')

len(data)